# Pandas do zero

Nada de arquivo grande, nada de JSON aninhado. Uma tabela de **8 linhas inventadas**,
onde você consegue conferir todo resultado contando nos dedos.

> ⚠️ Os dados aqui são **fictícios**. Os números não significam nada sobre LoL de
> verdade. O objetivo é só você enxergar o que cada comando faz.

Rode cada célula com `Shift + Enter`, na ordem.

In [1]:
import pandas as pd

---
## 1. Criando um DataFrame

Um **DataFrame** é uma tabela. A forma mais simples de criar um é a partir de um
dicionário, onde **cada chave vira uma coluna**:

In [2]:
dados = {
    "match_id":       ["M1", "M1", "M2", "M2", "M3", "M3", "M4", "M4"],
    "team_id":        [100,  200,  100,  200,  100,  200,  100,  200],
    "venceu":         [True, False, False, True, True, False, False, True],
    "primeiro_barao": [True, False, False, True, True, False, True, False],
    "primeira_torre": [True, False, True, False, False, True, True, False],
    "vision_score":   [210,  180,  195,  240,  160,  175,  220,  205],
}

df = pd.DataFrame(dados)
df

,match_id,team_id,venceu,primeiro_barao,primeira_torre,vision_score
0,M1,100,True,True,True,210
1,M1,200,False,False,False,180
2,M2,100,False,False,True,195
3,M2,200,True,True,False,240
4,M3,100,True,True,False,160
5,M3,200,False,False,True,175
6,M4,100,False,True,True,220
7,M4,200,True,False,False,205


Repare na coluna sem nome à esquerda, com 0 a 7. É o **índice** — o "número da
linha". O Pandas cria automaticamente.

Cada partida (M1, M2...) tem duas linhas: uma para cada time. É exatamente o grão
da sua futura tabela `times`.

In [3]:
print("shape (linhas, colunas):", df.shape)
print("colunas:", list(df.columns))

shape (linhas, colunas): (8, 6)
colunas: ['match_id', 'team_id', 'venceu', 'primeiro_barao', 'primeira_torre', 'vision_score']


---
## 2. Uma coluna: a Series

Pegar **uma** coluna devolve uma **Series** — que é uma coluna solta, com o índice
junto.

In [4]:
df["venceu"]

0     True
1    False
2    False
3     True
4     True
5    False
6    False
7     True
Name: venceu, dtype: bool

Pegar **várias** colunas devolve outro **DataFrame**. Repare no colchete duplo:
o de fora é "me dá isso", o de dentro é a lista de nomes.

In [5]:
df[["match_id", "venceu"]]

,match_id,venceu
0,M1,True
1,M1,False
2,M2,False
3,M2,True
4,M3,True
5,M3,False
6,M4,False
7,M4,True


### 🔹 Exercício 1

Na célula abaixo, mostre só as colunas `team_id` e `vision_score`.

In [6]:
df[["team_id", "vision_score"]]

,team_id,vision_score
0,100,210
1,200,180
2,100,195
3,200,240
4,100,160
5,200,175
6,100,220
7,200,205


---
## 3. Máscara booleana — a ideia central do Pandas

Quando você compara uma coluna com um valor, o Pandas compara **todas as linhas de
uma vez** e devolve uma Series de `True`/`False`, uma por linha:

In [7]:
df["vision_score"] > 200

0     True
1    False
2    False
3     True
4    False
5    False
6     True
7     True
Name: vision_score, dtype: bool

Confira olhando a tabela original: as linhas 0 (210), 3 (240), 6 (220) e 7 (205)
são as maiores que 200. Bate?

Essa Series de True/False se chama **máscara**. E quando você coloca uma máscara
dentro de `df[...]`, o Pandas **mantém só as linhas True**:

In [28]:
partidas = df[df["vision_score"] > 200]
partidas

,match_id,team_id,venceu,primeiro_barao,primeira_torre,vision_score,visao_alta
0,M1,100,True,True,True,210,True
3,M2,200,True,True,False,240,True
6,M4,100,False,True,True,220,True
7,M4,200,True,False,False,205,True


É isso que substitui o `for` do Python puro. Você não percorre nada — descreve a
condição, e ela vale para a coluna inteira.

Para colunas que **já são** True/False, nem precisa de comparação:

In [9]:
df[df["primeiro_barao"]]

,match_id,team_id,venceu,primeiro_barao,primeira_torre,vision_score
0,M1,100,True,True,True,210
3,M2,200,True,True,False,240
4,M3,100,True,True,False,160
6,M4,100,False,True,True,220


Para combinar condições: `&` significa E, `|` significa OU.

**Cada condição precisa estar entre parênteses.** Sem eles o Python erra a ordem
das operações e dá um erro confuso.

In [10]:
df[(df["primeiro_barao"]) & (df["venceu"])]

,match_id,team_id,venceu,primeiro_barao,primeira_torre,vision_score
0,M1,100,True,True,True,210
3,M2,200,True,True,False,240
4,M3,100,True,True,False,160


### 🔹 Exercício 2

Mostre só as linhas em que o time **pegou a primeira torre** E **perdeu**.

Dica: para negar uma coluna booleana, use o til: `~df["venceu"]`

In [12]:
df[(df["primeira_torre"]) & (~df["venceu"])]

,match_id,team_id,venceu,primeiro_barao,primeira_torre,vision_score
2,M2,100,False,False,True,195
5,M3,200,False,False,True,175
6,M4,100,False,True,True,220


---
## 4. O truque que vale ouro: True vale 1

No Pandas, `True` vale **1** e `False` vale **0**. Isso significa que:

- `.sum()` numa coluna booleana → **quantos** são True
- `.mean()` numa coluna booleana → **que proporção** é True

E "proporção de vitórias" é literalmente **taxa de vitória**.

In [13]:
print("times que venceram:", df["venceu"].sum())
print("taxa de vitória geral:", df["venceu"].mean())

times que venceram: 4
taxa de vitória geral: 0.5


Faz sentido: 8 times, 4 vitórias, 50%. Sempre 50% — em toda partida um time ganha
e o outro perde.

Agora junte as duas ideias e você responde o **PN01**:

In [14]:
# 1. filtra só quem pegou o primeiro Barão
com_barao = df[df["primeiro_barao"]]

# 2. calcula a taxa de vitória DESSE grupo
print("times que pegaram o primeiro Barão:", len(com_barao))
print("taxa de vitória deles:", com_barao["venceu"].mean())

times que pegaram o primeiro Barão: 4
taxa de vitória deles: 0.75


Confira na tabela: 4 times pegaram o primeiro Barão (linhas 0, 3, 4, 6) e desses,
3 venceram. 3/4 = 0,75.

**Duas linhas de Pandas responderam uma pergunta de negócio inteira.** Em Python
puro seriam uns 8 a 10 linhas com laços aninhados.

### 🔹 Exercício 3

Calcule a taxa de vitória dos times que pegaram a **primeira torre**.
Depois confira o resultado contando na tabela.

In [20]:
primeira_torre = df[df["primeira_torre"]]

print(f'Quantos times levaram a primeira torre: {len(primeira_torre)}')
print('Qual taxa de vitória dos times que levam a primeira torre:', primeira_torre["venceu"].mean())

Quantos times levaram a primeira torre: 4
Qual taxa de vitória dos times que levam a primeira torre: 0.25


---
## 5. Criando colunas novas

Atribuir a um nome que não existe **cria** a coluna. A operação vale para a coluna
inteira, sem laço:

In [21]:
df["visao_alta"] = df["vision_score"] > 200
df

,match_id,team_id,venceu,primeiro_barao,primeira_torre,vision_score,visao_alta
0,M1,100,True,True,True,210,True
1,M1,200,False,False,False,180,False
2,M2,100,False,False,True,195,False
3,M2,200,True,True,False,240,True
4,M3,100,True,True,False,160,False
5,M3,200,False,False,True,175,False
6,M4,100,False,True,True,220,True
7,M4,200,True,False,False,205,True


---
## 6. `groupby` — responder para vários grupos de uma vez

Até agora você filtrou um grupo e calculou. O `groupby` faz isso para **todos os
grupos ao mesmo tempo**.

In [25]:
# Taxa de vitória separada por: pegou o Barão x não pegou
df.groupby("vision_score")["venceu"].mean()

vision_score
160    1.0
175    0.0
180    0.0
195    0.0
205    1.0
210    1.0
220    0.0
240    1.0
Name: venceu, dtype: float64

Leia assim: *"agrupe pelas categorias de `primeiro_barao`, e em cada grupo calcule
a média de `venceu`"*.

O resultado responde o PN01 **completo** — com Barão e sem Barão — numa linha só.

Dá para pedir mais de uma medida ao mesmo tempo:

In [26]:
df.groupby("primeiro_barao")["venceu"].agg(["count", "sum", "mean"])

,count,sum,mean
primeiro_barao,,,
False,4,1,0.25
True,4,3,0.75


`count` = quantos times no grupo, `sum` = quantos venceram, `mean` = a taxa.

Sempre mostre o `count` junto da taxa. Lembra da lição L6 do charter? Uma taxa sem
o tamanho da base pode ser 90% de 40 partidas jogadas por 2 pessoas.

---
## Resumo

| Comando | O que faz |
|---|---|
| `pd.DataFrame(dicionario)` | cria uma tabela |
| `df.shape` | (linhas, colunas) |
| `df["col"]` | uma coluna (Series) |
| `df[["a", "b"]]` | várias colunas (DataFrame) |
| `df["col"] > 10` | máscara: True/False por linha |
| `df[mascara]` | mantém só as linhas True |
| `&` `\|` `~` | E, OU, NÃO (cada condição entre parênteses) |
| `df["col"].sum()` | soma (em booleano: quantos True) |
| `df["col"].mean()` | média (em booleano: a proporção) |
| `df["nova"] = ...` | cria coluna |
| `df.groupby("a")["b"].mean()` | média de `b` para cada valor de `a` |

Isso é praticamente tudo que o seu projeto precisa de Pandas.